In [1]:
import numpy as np
from scipy.integrate import simpson
import matplotlib.pyplot as plt

r = np.loadtxt('raw_rho_ch_40Ar.dat', delimiter=',', usecols=0) # radius (fm)
rho = np.loadtxt('raw_rho_ch_40Ar.dat', delimiter=',', usecols=1) # charge density (fm^-3)

R = [] # Store the r values (fm)
RHO = [] # Store the rho values (fm^-3)
i = 0

while (i < len(r)):
    rvals = r[r==r[i]] # Select values of r identical to a r[i]
    mean_rho = rho[r==r[i]] # Select all values of rho for r[i]
    mean_rho = np.mean(mean_rho, dtype=np.float64) # Compute the mean value of rho
    R.append(rvals[0])
    RHO.append(mean_rho)
    i += len(rvals)
R = np.array(R)
RHO = np.array(RHO)
    
with open("rho_ch_40Ar.dat", "w") as file:
    for x, y in zip(R, RHO):
        file.write(f"{x}\t{y}\n")

fac_conv = 1/0.1973 # 1 GeV = 1/0.1973 fm^-1
qmax = 5/fac_conv # maximum value of the momentum transfer (GeV)
q = np.linspace(0, qmax*fac_conv, 10000) # q values in fm^-1

NORM = simpson( R**2*RHO, x=R)

def ComputeFF(q):
    FF = np.zeros_like(q)
    for i in range(len(q)):
        if q[i] == 0:
            result = NORM
        else:
            result = simpson( RHO*R*np.sin(q[i]*R)/q[i] ,x=R)
        FF[i] = np.abs( result/NORM )
    return FF

FFvals = ComputeFF(q) # F(q^2) for q values (fm)
q /= fac_conv # q values in GeV

with open("40Ar_ChFF_NNLOsat_Barbieri.dat", "w") as file:
    for x, y in zip(q, FFvals):
        file.write(f"{x}\t{y}\n")